# 02 — Forecasting the intraday volatility curves

One-step-ahead forecast of the 2026-01-02 intraday curves from the FFM estimated in notebook `01`:
forecast the scalar factor score $\hat u_{1,T+1}$, forecast each idiosyncratic curve $\hat\xi_{i,T+1}(\tau)$,
then recombine $\hat x = \hat\mu_i + \sigma(\hat\chi + \hat\xi)$ on the original scale.

Requires the shared objects saved by `01_data_eda_ffm_rolling.ipynb` (`data/shared/`).

## Contents

1. [Imports & shared objects](#Imports-&-shared-objects)
2. [Stationarity](#Testing-stationarity)
3. [Forecasting factor](#forecasting-factor)
   - [Random Walk](#random-walk)
   - [AR(p)](#arp)
   - [ARIMA](#arima)
   - [ARFIMA](#arfima)
   - [HAR(p)](#harp)
   - [GARCH](#garch)
   - [Comparison](#Comparison-forecast)
4. [Forecasting the idiosyncratic component](#Forecasting-Idiosyncratic-component)
5. [Total forecast](#Forecast-total)
6. [Evaluation](#Forecast-evaluation)
7. [Training-period comparison](#Train-period-comparison)
8. [All-model comparison](#Comparison-Forecast-all-models)
9. [Robustness — frozen loading direction](#Robustness-—-degradation-of-a-frozen-loading-direction)

# Import

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import os
from functools import partial

In [ ]:
# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import acf

In [ ]:
# ── Data processing & factor model ───────────────────────────────────────────
from data_processing import resample_5min, log_returns_5min, trading_day_matrix, panel_to_dict
from ffm_transform import log_transform
from ffm_smoothing import smoothing_for_ffm, extract_smoothing_params
from ffm_fx_real import run_ffm, plot_raw_one_day
from factor_model_fcts import (
    build_panel_from_matrices, compute_daily_avg_vol, avg_vol_from_matrices,
    check_residual_common_structure, ffm_per_period,
)
from functions import plot_heatmap

In [ ]:
# ── EDA ───────────────────────────────────────────────────────────────────────
from ffm_eda import run_eda, missing_by_year_pair, get_stable_pairs
from ffm_eda_focused import run_focused_eda

In [ ]:
# ── Forecasting ───────────────────────────────────────────────────────────────
from ffm_forecasting import (
    test_adf, test_kpss, gph_test, test_arch,
    fractional_diff, test_stationarity,
    plot_acf_pacf,
    test_ljung_box, test_ljung_box_func,
    forecast_ar, forecast_arima, forecast_garch,
    har_regression, plot_forecast_har,
    select_garch_order, select_garch_variant,
    plot_all_forecasts,
    predict_operator_h, plot_forecast_idiosyncratic,
    reconstruct_forecast,
    plot_forecast_vs_observed,
    decompose_forecast_error,
    print_error_decomposition,
    plot_error_decomposition,
    compare_models_error,
    print_models_comparison,
    plot_models_comparison,
    select_p_far, forecast_idio_har,
    forecast_factor_rw, 
    forecast_idio_rw, run_model_comparison, 
    print_model_comparison, 
    plot_model_comparison,
    run_model_comparison_by_pair,
    print_model_comparison_by_pair,
    plot_model_comparison_by_pair, 
    forecast_all_assets_FAR,
    plot_comparison_periods,
    plot_idio_forecast_comparison,
    plot_reconstructed_forecast,
)

In [ ]:
# ── Stability ─────────────────────────────────────────────────────────────────
from ffm_stability import (
    grassman_rolling_window_ref, grassman_rolling_window,
    plot_grassman, plot_loadings_evolution,
    plot_loadings_mean_evolution, plot_r2_rolling,
    grassman_rolling_by_pair,
    analyze_negative_spikes,
    regression_grassmann_vol, plot_distance_distribution,
    loading_heatmap,
    compute_cusum, plot_cusum,
    bai_perron_distances, plot_bai_perron_distances,
    bai_perron_ffm, plot_bai_perron_ffm,
)

In [ ]:
%load_ext autoreload
%autoreload 2

# Imports & shared objects

Load the objects computed and saved by `01_data_eda_ffm_rolling.ipynb`.

In [ ]:
# ── Load shared objects (saved by 01_data_eda_ffm_rolling) ────────────────────
import joblib

shared_dir = "../data/shared/"

ffm               = joblib.load(f"{shared_dir}ffm.pkl")
vol_ffm_all       = joblib.load(f"{shared_dir}vol_ffm_all.pkl")
vol_matrices_all  = joblib.load(f"{shared_dir}vol_matrices_all.pkl")
vol_matrices_2026 = joblib.load(f"{shared_dir}vol_matrices_2026.pkl")
vol_ffm_2026      = joblib.load(f"{shared_dir}vol_ffm_2026.pkl")
avg_vol           = joblib.load(f"{shared_dir}avg_vol.pkl")
FX_all            = joblib.load(f"{shared_dir}FX_all.pkl")

U_hat      = ffm['U_hat']
pairs_name = ffm['pairs']
Xi_hat     = ffm['Xi_hat']

print(f"ffm loaded  — T={ffm['U_hat'].shape[1]} days, N={len(ffm['pairs'])} pairs")
print(f"vol_ffm_all — {len(vol_ffm_all)} pairs")

# Testing stationarity

## ADF & KPSS

In [ ]:
U_hat = ffm['U_hat']
test_stationarity(U_hat.T, 40)

## ACF & PACF

In [ ]:
plot_acf_pacf(U_hat.T)

## Ljung Box test

In [ ]:
test_ljung_box(U_hat.T)

## GPH test

In [ ]:
gph_t = gph_test(U_hat.T)

Shocks take a very long time to die out: they affect the future over a long horizon (long memory).

In [ ]:
d_hat  = gph_t['d_hat']
f1_fd  = fractional_diff(U_hat[0, :], d=0.5)

print("Fractionally differenced series:")
test_adf(f1_fd)
test_kpss(f1_fd)
gph_test(f1_fd)   # d̂ should be ≈ 0

In [ ]:
test_ljung_box(f1_fd)

In [ ]:
plot_acf_pacf(f1_fd)

# Forecasting factor

## Random Walk

In [ ]:
fc_rw, ci_l_rw, ci_u_rw = forecast_factor_rw(
    series  = ffm['U_hat'][0, :],
    horizon = [1, 2, 3, 4, 5],
    n_display = 50, 
    plot = True
)


## AR(p)

In [ ]:
result_ar, fc_ar, ci_l_ar, ci_u_ar = forecast_ar(U_hat.T, h=5, p_max= 20,n_display=50, plot= True)

In [ ]:
print(fc_ar)

## ARIMA

In [ ]:
result_arima, fc_arima, ci_low_arima, ci_high_arima = forecast_arima(U_hat.T,h=5,d=0, p_max=7, q_max=4,n_display=50, plot= True)

## ARFIMA

In [ ]:
result_arfima, fc_arfima, ci_low_arfima, ci_high_arfima = forecast_arima(f1_fd,h=5,d=0, p_max=7, q_max=4,n_display=50, plot= True)

## HAR(p)

In [ ]:
t = pd.DataFrame(ffm['U_hat'].T)
t.index = ffm['days']
t.columns = ['U_hat']
t['U_hat'].plot()

In [ ]:
t['U_hat'].hist(bins=50)
plt.title('Distribution of factor_1')
plt.show()

In [ ]:
acf_vals = acf(t['U_hat'], nlags=30)

print("Lag 1:", acf_vals[1])
print("Lag 5:", acf_vals[5])
print("Lag 22:", acf_vals[22])

In [ ]:
# shows that the 'daily', 'weekly' and 'monthly' terms are significant 

m = har_regression(ffm['U_hat'].T, summary=True)

In [ ]:
fc_har, ci_l_har, ci_u_har = plot_forecast_har(
    series    = U_hat[0, :],
    horizon   = [1, 2, 3, 4,5],
    n_display = 50, 
    plot= True
)

In [ ]:
print(fc_har)

In [ ]:
fc_har2, ci_l_har2, ci_u_har2 = plot_forecast_har(
    series    = U_hat[0, :],
    horizon   = [1], # must be a list
    n_display = 50, 
    plot= True
)

## GARCH

#### Test ARCH

In [ ]:
series = U_hat[0, :]
test_arch(series)

In [ ]:
series = U_hat[0, :]
result_garch, fc_mean, fc_std, ci= forecast_garch(series, 'AR',p=1,q=0,  h=5, plot=True, n_display=100, i=1)

In [ ]:
# ── APPLICATION ───────────────────────────────────────────────

series    = U_hat[0, :]

#res_arima = ARIMA(series, order=(1,0,0), trend='c').fit()
#resid     = res_arima.resid

# Order selection
best_pq = select_garch_order(resid, p_max=3, q_max=3)

# Compare the GARCH variants
results, best_name = select_garch_variant(resid)

## Comparison forecast

In [ ]:
# ── Application ───────────────────────────────────────────────

series = U_hat[0, :]

forecasts_dict = {
    'RW': fc_rw, 
    'AR' : fc_ar,
    'HAR': fc_har,
    #'ARFIMA': fc_arfima,
    'ARIMA': fc_arima
}

ci_dict = {
    'RW': (ci_l_rw, ci_u_rw),
    'AR' : (ci_l_ar,  ci_u_ar),
    'HAR': (ci_l_har, ci_u_har),
    #'ARFIMA': (ci_low_arfima, ci_high_arfima), 
    'ARIMA': (ci_low_arima, ci_high_arima)
}

plot_all_forecasts(series, forecasts_dict,
                   ci_dict=ci_dict, h=1, n_display=50, dates = ffm['days'])

# Forecasting Idiosyncratic component

### Test Ljung-Box functional 
Idiosyncratic component white noise ?

if test rejected, not white noise - FAR(1)

In [ ]:
pairs_name = ffm['pairs']
Xi_hat      = ffm['Xi_hat']

rejected =  test_ljung_box_func(Xi_hat, pairs_name,lags = [20,50], n_pc =5)

p-values = 0 for every pair $\Rightarrow$ the factor does not capture all the common dynamics

$\Rightarrow$ the idiosyncratic components still contain a lot of structure. 

- $d \in (0.3, 0.8)$
- strong long memory in the idiosyncratic components

$\Rightarrow$ fractional differencing 

In [ ]:
orders = select_p_far(
    Xi_hat     = ffm['Xi_hat'],   # (N, T, n_tau) — idiosyncratic components
    pairs_name = ffm['pairs'],    # list of pairs
    p_max      = 10,              # maximum order to test
    n_pc       = 5,               # number of PCs used for the BIC
)

print("\nSelected FAR orders:")
for pair, info in orders.items():
    print(f"  {pair:<12} → FAR({info['far_order']})   (p per PC: {info['p_per_cp']})")


### Random walk

In [ ]:
pred_rw, ci_l_rw, ci_h_rw = forecast_idio_rw(ffm['Xi_hat'], h=1)
# carry the last observation forward
plot_forecast_idiosyncratic(
    Xi_hat     = ffm['Xi_hat'],
    pairs_name = ffm['pairs'],
    h          = 1,
    forecasts  = pred_rw,
    title      = 'Idiosyncratic — Random Walk  h=1',
)

### FAR (FPCA + VAR)

In [ ]:
fc_far = forecast_all_assets_FAR(Xi_hat, ffm['pairs'])

plot_forecast_idiosyncratic(
    Xi_hat     = ffm['Xi_hat'],
    pairs_name = ffm['pairs'],
    h          = 1,
    forecasts  = fc_far,
    title      = 'Idiosyncratic — FAR (FPCA + VAR)  h=1',
)

### Operator $C_h$

In [ ]:
Xi_hat      = ffm['Xi_hat']

for h in [1,2,3,4,5]:
    fc_xi_op = predict_operator_h(
        Xi_hat     = ffm['Xi_hat'],
        sigma= ffm['sigma'],
        pairs_name = ffm['pairs'],
        h          = h,
        K          = 5
    )
    print(f"h={h} : forecasts shape = {fc_xi_op.shape}")  # (13, 288)

In [ ]:
h=1

fc_xi_op = predict_operator_h(
        Xi_hat     = ffm['Xi_hat'],
        sigma= ffm['sigma'],
        pairs_name = ffm['pairs'],
        h          = h,
        K          = 5
    )
pred_idio1 = plot_forecast_idiosyncratic(ffm['Xi_hat'], pairs_name, h=1, K=6, n_last=5, forecasts=fc_xi_op)

### HAR

In [ ]:
pred_idio_har, ci_low_har, ci_high_har = forecast_idio_har(
    Xi_hat     = ffm['Xi_hat'],
    pairs_name = ffm['pairs'],
    h          = 1,
    K          = 5,
    center_window = 252, 
    fit_window = 10
)

X_log_har, X_abs_har = reconstruct_forecast(fc_har, pred_idio_har, ffm)
#center_window = 252

In [ ]:
h=1
Xi_hat = ffm['Xi_hat']
pred_idio = plot_forecast_idiosyncratic(
    Xi_hat     = ffm['Xi_hat'],
    pairs_name = ffm['pairs'],
    h          = 1,
    forecasts  = pred_idio_har,
    title      = 'Idiosyncratic — HAR  h=1',
)

In [ ]:
ratio, corr = check_residual_common_structure(Xi_hat, pairs_name)

"The residual cross-sectional correlations in the idiosyncratic components (mean = 0.20) reflect the structural currency overlap inherent to FX pairs — all pairs share either USD or EUR as base or quote currency. This correlation is static and structural rather than dynamic, as evidenced by its persistence regardless of the number of factors K. It does not represent a failure of the FFM to capture common dynamics, but rather an irreducible feature of FX market microstructure. The single-factor specification is therefore retained, consistent with the spectral ratio criterion."

In [ ]:
from ffm_forecasting import plot_idio_forecast_comparison

plot_idio_forecast_comparison(
    ffm = ffm,   # or ffm_2025
    forecasts_dict = {
        'RW'  : (pred_rw,       ci_l_rw,     ci_h_rw),
        'FAR' : (fc_far,        None,         None),      # no CI for FAR
        'HAR' : (pred_idio_har, ci_low_har,   ci_high_har),
    },
    pairs_to_plot = ['eurusd', 'audjpy', 'eurchf', 'usdtry'],
    n_last_obs    = 10,
    h             = 1,
)


# Forecast total

### Smoothing

In [ ]:
target_date = pd.Timestamp('2026-01-02')

pairs       = ffm['pairs']
tau         = ffm['tau']
N           = len(pairs)

In [ ]:
h=1
X_log, X_abs = reconstruct_forecast(fc_ar, pred_idio_har, ffm)
print(f"h={h} : {X_abs.shape}")

In [ ]:
plot_forecast_vs_observed(
    target_date       = '2026-01-02',
    ffm               = ffm,
    #X_log             = X_log,
    X_abs= X_abs,
    vol_matrices_2026 = vol_matrices_2026,
    vol_ffm_2026      = vol_ffm_2026,
    h                 = 1,
)


**Evaluation target — two options**

- *Option 1*: smooth the 2026 data before comparing. The model is then evaluated on smoothed data, i.e. on how well it forecasts the smooth structure.
- *Option 2*: compare in log space. The log compresses the noise, which makes the comparison more readable.
  - compute MSFE and MAFE
  - comparing in log space is more rigorous and standard in the literature 

In [ ]:
from ffm_forecasting import plot_reconstructed_forecast

plot_reconstructed_forecast(
    ffm  = ffm,   # or ffm_2025
    models_dict = {
        'AR + HAR' : (fc_ar,pred_idio_har),
        'AR + FAR' : (fc_ar,fc_far),
        'AR + RW'  : (fc_ar,pred_rw),
    },
    target_date       = '2026-01-02',
    vol_ffm_2026      = vol_ffm_2026,
    vol_matrices_2026 = vol_matrices_2026,
    pairs_to_plot     = ['eurusd', 'audjpy', 'eurchf', 'usdtry'],
    h                 = 1,
)


# Forecast evaluation

In [ ]:
df_err = decompose_forecast_error(
    target_date       = '2026-01-02',
    ffm               = ffm,
    X_log             = X_log,
    vol_matrices_2026 = vol_matrices_2026,
    vol_ffm_2026      = vol_ffm_2026,
    exclude_pairs     = ['usdcnh'],
)


In [ ]:
print_error_decomposition(df_err)
plot_error_decomposition(df_err, metric='MAFE', title_suffix='2026-01-02')
plot_error_decomposition(df_err, metric='MSFE', title_suffix='2026-01-02')


# Train period comparison

Compare forecasts from two training windows:
- **Long**: 2014–2025 (model `ffm`, already estimated)
- **Short**: 2025 only (`ffm_2025`)

In [ ]:
ffm_2025 = run_ffm(
    vol_ffm_all,
    k_max        = 5,
    min_coverage = 0.7,
    date_start   = '2025-07-01',
    date_end     = '2025-12-31',
    silent       = True,
)


In [ ]:
# ── 2. Forecast factors — HAR on 2025 model ──────────────────────────────────
U_hat_2025 = ffm_2025['U_hat']

_, fc_arima_2025, ci_low_arima_2025, ci_high_arima_2025 = forecast_arima(U_hat_2025.T,
                                                          h=1,
                                                          d=0, 
                                                          p_max=7, 
                                                          q_max=4,
                                                          n_display=50, plot= False)

#fc_har_2025, ci_l_2025, ci_u_2025 = plot_forecast_har(
#    series    = U_hat_2025[0, :],
#    horizon   = [1, 2, 3, 4, 5],
#    n_display = 50,
#    plot      = False,
#)


In [ ]:
pred_idio_har_2025, ci_low_har_2025, ci_high_har_2025 = forecast_idio_har(
    Xi_hat     = ffm_2025['Xi_hat'],
    pairs_name = ffm_2025['pairs'],
    h          = 1,
    K          = 5,
    #center_window = 252, 
    #fit_window = 10
)

pred_idio_2025 = plot_forecast_idiosyncratic(
    ffm_2025['Xi_hat'],
    ffm_2025['pairs'],
    h      = 1,
    K      = 5,
    n_last = 5,
    forecasts= pred_idio_har_2025
)


In [ ]:
# ── 4. Reconstruct full curve forecast ───────────────────────────────────────
X_log_2025, X_abs_2025 = reconstruct_forecast(fc_arima_2025, pred_idio_2025, ffm_2025)
print(f'X_abs_2025 shape : {X_abs_2025.shape}')


In [ ]:
plot_forecast_vs_observed(
    target_date       = '2026-01-02',
    ffm               = ffm_2025,
    X_abs             = X_abs_2025,
    vol_matrices_2026 = vol_matrices_2026,
    vol_ffm_2026      = vol_ffm_2026,
    h                 = 1,
)


In [ ]:
df_err_2025 = decompose_forecast_error(
    target_date       = '2026-01-02',
    ffm               = ffm_2025,
    X_log             = X_log_2025,
    vol_matrices_2026 = vol_matrices_2026,
    vol_ffm_2026      = vol_ffm_2026,
    exclude_pairs     = ['usdcnh'],
)

print_error_decomposition(df_err_2025)
plot_error_decomposition(df_err_2025, metric='MAFE', title_suffix='2026-01-02 S2_2025')
plot_error_decomposition(df_err_2025, metric='MSFE', title_suffix='2026-01-02 S2_2025')

### Error comparison across training periods

In [ ]:
#target_dates = ['2026-01-02', '2026-01-05', '2026-01-06',
#                '2026-01-07', '2026-01-08', '2026-01-09']

target_dates = ['2026-01-02']

fc_rw_bench, _, _ = forecast_factor_rw(
    series  = ffm['U_hat'][0, :],
    horizon = [1],
    plot    = False,
)
pred_idio_rw_bench, _, _ = forecast_idio_rw(ffm['Xi_hat'], h=1)
X_log_rw, X_rw = reconstruct_forecast(fc_rw_bench[0], pred_idio_rw_bench, ffm)

df_compare = compare_models_error(
    target_dates      = target_dates,
    models            = {
        '2014-2025' : (ffm,      X_log),
        '2025'      : (ffm_2025, X_log_2025),
        'RW'        : (ffm,      X_log_rw),
    },
    vol_matrices_2026 = vol_matrices_2026,
    vol_ffm_2026      = vol_ffm_2026,
    exclude_pairs     = ['usdcnh'],
)

print_models_comparison(df_compare)


In [ ]:
plot_models_comparison(df_compare, metric='MAFE',
                       title_suffix='Jan 2026')
plot_models_comparison(df_compare, metric='MSFE',
                       title_suffix='Jan 2026')

# Comparison Forecast all models 

### 2014-2025

In [ ]:
df_cmp = run_model_comparison(
    target_dates      = ['2026-01-02', '2026-01-03'],   # h=1 → jan 2, h=2 → jan 3
    ffm               = ffm,
    vol_matrices_2026 = vol_matrices_2026,
    vol_ffm_2026      = vol_ffm_2026,
    horizons          = [1,2],
    models_factor     = [ 'RW', 'AR'],
    models_idio       = ['RW', 'FAR', 'HAR'],
    exclude_pairs     = ['usdcnh'],
)


In [ ]:

# Export as CSV
df_cmp.to_csv('../data/results/model_comparison.csv')

# Export for comparison with the raw-data pipeline (04_raw_ffm)
joblib.dump(df_cmp, f"{shared_dir}df_cmp_smooth.pkl")


In [ ]:
plot_model_comparison(df_cmp, 'all', h=1)

#### By pair

In [ ]:
df_cmp_pair = run_model_comparison_by_pair(
    target_dates      = ['2026-01-02', '2026-01-03'],
    ffm               = ffm,
    vol_matrices_2026 = vol_matrices_2026,
    vol_ffm_2026      = vol_ffm_2026,
    horizons          = [1, 2],
    models_factor     = ['RW', 'AR', 'HAR', 'ARIMA'],
    models_idio       = ['RW', 'FAR', 'HAR'],
    exclude_pairs     = ['usdcnh'],
)

In [ ]:
plot_model_comparison_by_pair(df_cmp_pair, 'all', h=1)


In [ ]:
from ffm_forecasting import plot_model_bar_by_pair

plot_model_bar_by_pair(df_cmp_pair, period='all', h=1)





### 2025

In [ ]:
df_cmp_2025 = run_model_comparison(
    target_dates      = ['2026-01-02', '2026-01-03'],   # h=1 → jan 2, h=2 → jan 3
    ffm               = ffm_2025,
    vol_matrices_2026 = vol_matrices_2026,
    vol_ffm_2026      = vol_ffm_2026,
    horizons          = [1, 2],
    models_factor     = ['RW', 'AR', 'HAR', 'ARIMA'],
    models_idio       = ['RW', 'FAR', 'HAR'],
    exclude_pairs     = ['usdcnh'],
)


In [ ]:
plot_model_comparison(df_cmp_2025, '2025_S2', h=1)

In [ ]:
df_cmp_pair_2025 = run_model_comparison_by_pair(
    target_dates      = ['2026-01-02', '2026-01-03'],
    ffm               = ffm_2025,
    vol_matrices_2026 = vol_matrices_2026,
    vol_ffm_2026      = vol_ffm_2026,
    horizons          = [1, 2],
    models_factor     = ['RW', 'AR', 'HAR', 'ARIMA'],
    models_idio       = ['RW', 'FAR', 'HAR'],
    exclude_pairs     = ['usdcnh'],
)

In [ ]:
plot_model_comparison_by_pair(df_cmp_pair_2025, period='2025', h=1)

In [ ]:
col = ('MAFE_raw', 'h=1')   # or 'h=2'

# Single benchmark: RW×RW of the full model (or of 2025 — same target either way)
rw_ref = df_cmp.loc[('RW', 'RW'), col]

compare = pd.DataFrame({
    'Full 2024–25' : (df_cmp[col]      / rw_ref - 1) * 100,
    '2025 S2 only' : (df_cmp_2025[col] / rw_ref - 1) * 100,
})
# Δ > 0 → 2025 S2 makes MORE error → the long training period wins
# Δ < 0 → 2025 S2 makes LESS error → the short training period wins
compare['Δ (S2 – Full)'] = compare['2025 S2 only'] - compare['Full 2024–25']

print(f"RW×RW reference MAFE_raw (h=1): {rw_ref:.4e}\n")
print(compare.round(2).to_string())

In [ ]:
# Comparison: Full 2024-25 vs 2025 S2 — same RW×RW baseline (Full)
plot_comparison_periods(df_cmp, df_cmp_2025, h=1)


# Robustness — degradation of a frozen loading direction

For each six-month window $w$ of the rolling grid of Chapter 5, we form a one-step forecast of the common component twice: once with the leading loading direction estimated on window $w$ (the up-to-date direction), once with the direction of the January–June 2014 reference window (the frozen direction). In both cases the score is carried forward as a random walk, which Section 6.1 showed captures its one-step predictable content, and the mean curve, the scale and the idiosyncratic part are identical. The only thing that differs between the two forecasts is the orientation of the loading subspace. We record the degradation

$$\Delta_w = \mathrm{MSFE}_{\text{frozen}}(w) - \mathrm{MSFE}_{\text{fresh}}(w)$$

on the first day outside the window.

> The last cells of this section relate $\Delta_w$ to the Grassmann distances computed in `03_stability.ipynb`
> (`data/results/grass_dist_6_30_mu.parquet`): run notebook `03` first.

In [ ]:
from ffm_forecasting import (
    forecast_degradation_frozen_direction,
    plot_forecast_degradation,
)

In [ ]:
df_degradation = forecast_degradation_frozen_direction(
    vol_ffm_all,
    window_size = 6,
    step        = 30,
    start_date  = '2014-01-01',
    end_date    = '2025-07-01',
)

df_degradation.to_parquet(
    "../data/results/degradation_frozen_vs_fresh_6_30.parquet",
    engine='pyarrow',
)
df_degradation.tail()

In [ ]:
plot_forecast_degradation(df_degradation, window_size=6, step=30)

In [ ]:
print(f"Mean   Δ_w           = {df_degradation['delta'].mean():.4e}")
print(f"Median Δ_w           = {df_degradation['delta'].median():.4e}")
print(f"Share of windows with Δ_w > 0 (frozen worse) = {(df_degradation['delta'] > 0).mean():.1%}")

In [ ]:
dg_ref_6M_30d = pd.read_parquet("../data/results/grass_dist_6_30_mu.parquet", engine='pyarrow')

In [ ]:
print(dg_ref_6M_30d.head())

In [ ]:
temp1= dg_ref_6M_30d['distance_grass']
temp2= df_degradation['delta']

correlation = temp1.corr(temp2)
print(correlation)
import statsmodels.api as sm
X = sm.OLS(temp2, sm.add_constant(temp1))
results = X.fit()
print(results.summary())